In [2]:
pip install tiktoken -q

Note: you may need to restart the kernel to use updated packages.


In [3]:
import tiktoken

text = """
Alembic — это инструмент для управления миграциями базы данных в SQLAlchemy.
Он позволяет безопасно обновлять схему БД при изменении моделей.
"""

enc = tiktoken.encoding_for_model("gpt-3.5-turbo")
tokens = enc.encode(text)
print(len(tokens))

52


In [42]:
def add_metadata(chunks: list[str], source: str, doc_id: str) -> list[dict]:
    if not (source and doc_id and chunks and all(chunks)):
        raise ValueError()
    
    return [{"text": chunk, "source": source, "doc_id": doc_id} for chunk in chunks]

In [44]:
add_metadata(["Чанк 1", "Чанк 2"], "wiki", "DOC-001")

[{'text': 'Чанк 1', 'source': 'wiki', 'doc_id': 'DOC-001'},
 {'text': 'Чанк 2', 'source': 'wiki', 'doc_id': 'DOC-001'}]

---

### Reranking: место в пайплайне и практическое применение

**Векторный поиск** находит фрагменты по **эмбеддинговому сходству** — но это не всегда совпадает с тем, что нужно пользователю.

**Пример:**
```
Запрос: "Как откатить миграцию в Alembic?"

Векторный поиск (топ-3 по косинусному сходству):
1. "Alembic — инструмент для управления миграциями БД."  ← семантически близко, но не ответ
2. "Команда alembic downgrade позволяет откатить миграцию."  ← точный ответ, но вектор далёк
3. "Миграции применяются через alembic upgrade head."  ← похоже, но не то
```

**Reranker** «перечитывает» запрос и каждый фрагмент вместе, оценивая **реальную релевантность:**
```
После reranking:
1. "Команда alembic downgrade позволяет откатить миграцию."  ✓
2. "Миграции применяются через alembic upgrade head."
3. "Alembic — инструмент для управления миграциями БД."
```
*Reranking улучшает точность ответов RAG на 15–30% — за счёт более тонкой оценки семантической релевантности.

#### Практика: reranking с flashrank

In [46]:
pip install flashrank -q

Note: you may need to restart the kernel to use updated packages.


In [47]:
from flashrank import Ranker, RerankRequest

# 1. Инициализируем reranker (модель загрузится автоматически)
ranker = Ranker()  # по умолчанию: "ms-marco-TinyBERT-L-2-v2"
# ranker = Ranker(model_name="ms-marco-MultiBERT-L-12") # лучше для мультиязычных задач

# 2. Готовим данные: запрос + фрагменты от векторного поиска
query = "Как откатить миграцию в Alembic?"
passages = [
    {"id": 0, "text": "Alembic — инструмент для управления миграциями БД."},
    {"id": 1, "text": "Команда alembic downgrade позволяет откатить миграцию."},
    {"id": 2, "text": "Миграции применяются через alembic upgrade head."},
]

# 3. Запускаем reranking
request = RerankRequest(query=query, passages=passages)
results = ranker.rerank(request)

print(results)

INFO:flashrank.Ranker:Downloading ms-marco-TinyBERT-L-2-v2...
ms-marco-TinyBERT-L-2-v2.zip: 100%|██████████| 3.26M/3.26M [00:00<00:00, 3.93MiB/s]

[{'id': 1, 'text': 'Команда alembic downgrade позволяет откатить миграцию.', 'score': np.float32(0.9999722)}, {'id': 0, 'text': 'Alembic — инструмент для управления миграциями БД.', 'score': np.float32(0.99996257)}, {'id': 2, 'text': 'Миграции применяются через alembic upgrade head.', 'score': np.float32(0.9999553)}]


**Производительность:** flashrank работает на CPU, обрабатывает 10 фрагментов за ~100–300 мс — достаточно для интерактивных приложений.

### Что делать, если фрагмент не влезает в контекст reranker

Большинство reranker-моделей имеют лимит контекста (~512 токенов).

**Решение — на этапе чанкинга (до векторизации!):**

1) **Дробите документы на чанки ≤400 токенов** (с overlap 50) — ещё до того, как они попадут в векторную базу.
2) **Не передавайте в reranker «сырые» длинные фрагменты** — только те, что уже прошли чанкинг.
3) **Если чанк всё же длинный — обрежьте его до лимита модели:**

In [48]:
from flashrank import Ranker, RerankRequest

ranker = Ranker()
MAX_TOKENS = 512  # лимит модели
CHARS_PER_TOKEN = 4  # грубая оценка для русского

def safe_rerank(query: str, passages: list[dict]) -> list:
    """Reranking с защитой от переполнения контекста."""
    # Обрезаем тексты до лимита (по символам, для простоты)
    trimmed = [
        {**p, "text": p["text"][:MAX_TOKENS * CHARS_PER_TOKEN]}
        for p in passages
    ]
    request = RerankRequest(query=query, passages=trimmed)
    return ranker.rerank(request)

### Нужно ли дообучать reranker на своих данных

**Короткий ответ:** на старте — нет. Дообучение оправдано только при специфичных доменах и больших объёмах размеченных данных.

**Когда можно рассмотреть дообучение:**

* У вас есть ≥1000 пар «запрос — релевантный фрагмент» с ручной разметкой.
* Базовая модель систематически ошибается на ваших типах запросов.
* Вы уже измерили прирост от reranking и хотите выжать ещё +5–10%.

**Почему не стоит дообучать на старте:**

* Требует размеченных данных (дорого и долго собирать).
* Риск переобучения на узком домене.
* Базовые модели (TinyBERT, MultiBERT) уже хорошо работают на общих задачах.
 

### Когда использовать reranking (а когда — нет)

**Используйте reranking, если:**

* Вы берёте из векторного поиска >=3 кандидата (top-10, top-20).
* Точность ответа критична (поддержка, юридические вопросы, техническая документация).
* У вас есть запас по времени: +100–300 мс на запрос допустимы.
* Вы уже настроили чанкинг и метаданные — reranking даст максимальный эффект на подготовленных данных.

**Можно отложить reranking, если:**

* Вы на этапе прототипа — сначала добейтесь работы базового RAG.
* Запросов мало, а точность не критична (внутренний поиск по заметкам).
* Бюджет по времени жёсткий (<500 мс на весь запрос).

! Стратегия старта: начните без reranking → измерьте точность → добавьте reranking → сравните результат. Так вы поймёте, окупается ли сложность в вашем случае.

### Чеклист: reranking в продакшене

1) Настройте чанкинг: фрагменты ≤400 токенов, overlap 50.
2) Добавьте метаданные: source, doc_id, created_at — минимум.
3) Векторный поиск: берите top-10–20 кандидатов (не top-3!).
4) Фильтрация по метаданным — до reranking (в векторной базе).
5) Reranking: flashrank
6) Передавайте в LLM только top-3 после reranking.
7) Логируйте score reranking — это поможет отлаживать точность.
 

### Полезно знать
**Примечание**
* FlashRank кэширует модели в /tmp/ — первая загрузка займёт время, последующие — мгновенно.
* Score reranking не нормирован между разными запросами — сравнивайте score только внутри одного запроса.
* Тестируйте reranking на реальных запросах — синтетические данные могут дать ложное ощущение точности.